In [52]:
# Text-to-Speech for Web Content (Simple Version)

In [1]:
!pip install crawl4ai pyttsx3 nest_asyncio beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import asyncio
from crawl4ai import AsyncWebCrawler
from bs4 import BeautifulSoup
import pyttsx3

In [3]:
# Helper to run crawl4ai on Windows Jupyter
import threading

def crawl_url(url, max_chars=1000):
    result_text = [None]
    error = [None]
    def _run():
        import asyncio
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)
        async def _crawl():
            async with AsyncWebCrawler(verbose=True) as crawler:
                res = await crawler.arun(url=url)
                # Parse HTML and extract only text
                soup = BeautifulSoup(res.html, 'html.parser')
                # Remove script and style tags
                for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                    tag.decompose()
                plain_text = soup.get_text(separator=' ', strip=True)
                return plain_text[:max_chars]
        try:
            result_text[0] = loop.run_until_complete(_crawl())
        except Exception as e:
            error[0] = e
        finally:
            loop.close()
    t = threading.Thread(target=_run)
    t.start()
    t.join()
    if error[0]:
        raise error[0]
    return result_text[0]

In [4]:
# List all voices
engine = pyttsx3.init()
voices = engine.getProperty('voices')

for idx, voice in enumerate(voices):
    print(f"{idx}: {voice.name}")
    
engine.stop()

0: Microsoft David Desktop - English (United States)
1: Microsoft Zira Desktop - English (United States)


In [5]:
# Extract text from URL using crawl4ai
url = "https://tinygrad.org/#faq"
text = crawl_url(url, max_chars=1000)

print(f"Extracted {len(text)} characters")
print(text[:200])

[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://tinygrad.org/#faq                                                                            |
✓ | ⏱: 0.66s 

[SCRAPE].. ◆ https://tinygrad.org/#faq                                                                            |
✓ | ⏱: 0.03s 

[COMPLETE] ● https://tinygrad.org/#faq                                                                            |
✓ | ⏱: 0.71s 

Extracted 1000 characters
tinygrad: A simple and powerful neural network framework tinygrad | docs | jobs | tinybox (buy now!) | FAQ tinygrad We write and maintain tinygrad , the fastest growing neural
    network framework It


In [7]:
# Convert to speech
engine = pyttsx3.init()
engine.say(text)
engine.runAndWait()
engine.stop()
#saving the audio to a file
engine.save_to_file(text, 'output.mp3')
engine.runAndWait()

In [8]:
# FAST speech (250 wpm)
engine = pyttsx3.init()
engine.setProperty('rate', 250)
engine.say(text[:400])
engine.runAndWait()
engine.stop()
#saving the audio to a file
engine.save_to_file(text[:400], 'output_fast.mp3')
engine.runAndWait()

In [9]:
# SLOW speech (100 wpm)
engine = pyttsx3.init()
engine.setProperty('rate', 100)
engine.say(text[:400])
engine.runAndWait()
engine.stop()
#saving the audio to a file
engine.save_to_file(text[:400], 'output_slow.mp3')
engine.runAndWait()

In [10]:
# Change voice (set voice_index to 0, 1, 2, etc)
voice_index = 0

engine = pyttsx3.init()
voices = engine.getProperty('voices')
engine.setProperty('voice', voices[voice_index].id)
print(f"Using: {voices[voice_index].name}")

engine.say(text[:400])
engine.runAndWait()
engine.stop()
#saving the audio to a file
engine.save_to_file(text[:400], f'output_voice_{voice_index}.mp3')
engine.runAndWait()

Using: Microsoft David Desktop - English (United States)
